# Actividad 1: modelos de optimización con Python

**Módulo I — Modelación y optimización**

Esta actividad contiene los dos ejemplos presentados en las imágenes: un problema de inversión resuelto mediante programación lineal y un problema del agente viajero resuelto mediante programación dinámica.

## Resultados de aprendizaje

- Identificar variables de decisión, función objetivo y restricciones.
- Formular y resolver un modelo lineal con PuLP.
- Representar un problema de rutas mediante una matriz de costos.
- Interpretar una solución óptima en el contexto del problema.
- Analizar el efecto de modificar parámetros y restricciones.

## Preparación del entorno

Los ejemplos requieren `pulp` y `python-tsp`. La siguiente celda instala las bibliotecas únicamente cuando no están disponibles.

In [ ]:
try:
    import pulp
except ImportError:
    %pip install -q pulp
    import pulp

try:
    import python_tsp
except ImportError:
    %pip install -q python-tsp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pulp import LpMaximize, LpProblem, LpStatus, LpVariable, PULP_CBC_CMD, value
from python_tsp.exact import solve_tsp_dynamic_programming

# Ejemplo 1. Selección óptima de inversiones

Una persona dispone de hasta 50 000 unidades monetarias para invertir en bonos y acciones. Debe invertir por lo menos 20 000 en bonos y no puede invertir más de 15 000 en acciones. La rentabilidad de los bonos es 0,25 % y la de las acciones es 4,5 %.

## Formulación

Variables de decisión:

- $x$: cantidad invertida en bonos.
- $y$: cantidad invertida en acciones.

Función objetivo:

$$\max Z=0.0025x+0.045y.$$

Restricciones:

$$x+y\leq 50000,$$
$$x\geq 20000,$$
$$0\leq y\leq 15000.$$

In [ ]:
modelo = LpProblem(name='problema-inversion', sense=LpMaximize)

x = LpVariable(name='bonos', lowBound=20_000, cat='Continuous')
y = LpVariable(name='acciones', lowBound=0, upBound=15_000, cat='Continuous')

modelo += 0.0025 * x + 0.045 * y, 'Rentabilidad total'
modelo += x + y <= 50_000, 'Presupuesto total'

solver = PULP_CBC_CMD(msg=False)
modelo.solve(solver)

x_valor = x.value()
y_valor = y.value()
rentabilidad = value(modelo.objective)

print(f'Estado de la solución: {LpStatus[modelo.status]}')
print(f'Inversión óptima en bonos: {x_valor:,.2f}')
print(f'Inversión óptima en acciones: {y_valor:,.2f}')
print(f'Rentabilidad total máxima: {rentabilidad:,.2f}')

## Interpretación del resultado

Como las acciones ofrecen la mayor rentabilidad, el modelo asigna a este activo el máximo permitido: 15 000. El presupuesto restante, 35 000, se invierte en bonos. La solución utiliza todo el presupuesto y produce una rentabilidad total de 762,5 unidades monetarias.

In [ ]:
resumen_inversion = pd.DataFrame({
    'Activo': ['Bonos', 'Acciones'],
    'Inversión óptima': [x_valor, y_valor],
    'Tasa': [0.0025, 0.045],
    'Rentabilidad': [0.0025 * x_valor, 0.045 * y_valor],
})
display(resumen_inversion)

fig, ejes = plt.subplots(1, 2, figsize=(11, 4))
ejes[0].bar(resumen_inversion['Activo'], resumen_inversion['Inversión óptima'],
            color=['#2878B5', '#F28E2B'])
ejes[0].set(ylabel='Unidades monetarias', title='Distribución óptima del presupuesto')
ejes[0].grid(axis='y', alpha=0.25)

ejes[1].bar(resumen_inversion['Activo'], resumen_inversion['Rentabilidad'],
            color=['#2878B5', '#F28E2B'])
ejes[1].set(ylabel='Unidades monetarias', title='Rentabilidad por activo')
ejes[1].grid(axis='y', alpha=0.25)
plt.tight_layout()
plt.show()

## Análisis de escenarios del ejemplo 1

Modifique el límite máximo de inversión en acciones y observe cómo cambia la solución.

In [ ]:
def resolver_inversion(maximo_acciones, presupuesto=50_000):
    problema = LpProblem('escenario-inversion', LpMaximize)
    bonos = LpVariable('bonos', lowBound=20_000)
    acciones = LpVariable('acciones', lowBound=0, upBound=maximo_acciones)
    problema += 0.0025 * bonos + 0.045 * acciones
    problema += bonos + acciones <= presupuesto
    problema.solve(PULP_CBC_CMD(msg=False))
    return {
        'máximo en acciones': maximo_acciones,
        'bonos': bonos.value(),
        'acciones': acciones.value(),
        'rentabilidad': value(problema.objective),
    }

escenarios_inversion = pd.DataFrame(
    [resolver_inversion(limite) for limite in [5_000, 10_000, 15_000, 20_000, 25_000]]
)
escenarios_inversion

# Ejemplo 2. Problema del agente viajero (TSP)

Un proceso de pintura debe visitar cuatro estaciones identificadas por el color de pintura: Blanca, Amarilla, Negra y Roja. El costo de desplazarse entre estaciones se representa mediante la siguiente matriz. El objetivo es encontrar una ruta cerrada que visite cada estación exactamente una vez y minimice la distancia total.

La matriz es **asimétrica**: viajar de Blanca a Amarilla cuesta 10, mientras que regresar de Amarilla a Blanca cuesta 20. Por tanto, la dirección de la ruta modifica el costo.

In [ ]:
nombres = ['Blanca', 'Amarilla', 'Negra', 'Roja']
matriz_distancias = np.array([
    [0, 10, 17, 15],
    [20, 0, 19, 18],
    [50, 44, 0, 22],
    [45, 40, 20, 0],
])

tabla_distancias = pd.DataFrame(
    matriz_distancias, index=nombres, columns=nombres
)
tabla_distancias.index.name = 'Pintura'
tabla_distancias

In [ ]:
ruta, distancia = solve_tsp_dynamic_programming(matriz_distancias)
ruta_cerrada = ruta + [ruta[0]]
ruta_nombres = [nombres[i] for i in ruta_cerrada]

print('Ruta óptima:', ' → '.join(ruta_nombres))
print('Índices para presentación:', [i + 1 for i in ruta_cerrada])
print(f'Distancia total: {distancia}')

### Corrección aplicada al código de las imágenes

La librería devuelve índices desde 0. Para mostrar una numeración desde 1 puede utilizarse `[i + 1 for i in ruta]`, pero la ruta original debe conservarse para acceder a las coordenadas y a la matriz. Sumar 1 directamente al arreglo y utilizarlo después como índice produciría posiciones inválidas.

In [ ]:
coordenadas = np.array([
    [0, 0],    # Blanca
    [10, 10],  # Amarilla
    [20, 5],   # Negra
    [15, 15],  # Roja
])

def graficar_ruta(coordenadas, ruta, etiquetas):
    ruta_cerrada = ruta + [ruta[0]]
    puntos = coordenadas[ruta_cerrada]

    plt.figure(figsize=(9, 6))
    plt.scatter(coordenadas[:, 0], coordenadas[:, 1], s=120, color='#2878B5', zorder=3)
    for i, etiqueta in enumerate(etiquetas):
        plt.text(coordenadas[i, 0] + 0.4, coordenadas[i, 1] + 0.4, etiqueta, fontsize=11)

    for origen, destino in zip(puntos[:-1], puntos[1:]):
        plt.annotate('', xy=destino, xytext=origen,
                     arrowprops={'arrowstyle': '->', 'color': '#D84315', 'lw': 2})

    plt.xlabel('Coordenada X')
    plt.ylabel('Coordenada Y')
    plt.title(f'Mejor recorrido del agente viajero — distancia {distancia}')
    plt.grid(alpha=0.3)
    plt.axis('equal')
    plt.tight_layout()
    plt.show()

graficar_ruta(coordenadas, ruta, nombres)

## Interpretación del ejemplo 2

La ruta óptima es Blanca → Amarilla → Negra → Roja → Blanca, con una distancia total de 96. El algoritmo compara combinaciones posibles y selecciona la de menor costo. En problemas grandes, el número de alternativas crece rápidamente; por ello, el TSP permite discutir complejidad computacional y la necesidad de métodos exactos, heurísticos o aproximados.

# Actividad que debe entregar el estudiante

## Parte A. Inversiones

1. Identifique variables, objetivo y restricciones.
2. Compruebe la solución óptima y explique por qué las acciones alcanzan su límite superior.
3. Cambie la rentabilidad de los bonos a 5 % y resuelva nuevamente.
4. Compare límites máximos de acciones de 5 000, 15 000 y 25 000.
5. Formule una nueva restricción de diversificación y analice el resultado.

## Parte B. Agente viajero

1. Verifique manualmente el costo de la ruta óptima.
2. Explique por qué la matriz es asimétrica y cómo afecta la solución.
3. Modifique al menos tres distancias y determine la nueva ruta.
4. Agregue una quinta estación con sus costos de entrada y salida.
5. Contextualice el problema como rutas de reparto, inspección o mantenimiento.

## Entrega

Presente el notebook ejecutado con formulación matemática, código, resultados, gráficas, comparación de escenarios y conclusiones propias.

# Conclusiones orientadoras

- La programación lineal convierte una decisión de asignación de recursos en variables, objetivo y restricciones verificables.
- Una solución óptima debe interpretarse en el contexto; el valor numérico por sí solo no explica su utilidad ni sus limitaciones.
- En el TSP, la solución depende de todos los costos de transición y no solamente de escoger el destino más cercano en cada paso.
- El análisis de escenarios permite evaluar sensibilidad y reconocer qué restricciones controlan la solución.
- El crecimiento combinatorio del TSP evidencia que el costo computacional también debe considerarse al seleccionar un método de solución.

# Criterios de evaluación

| Criterio | Porcentaje |
|---|---:|
| Formulación matemática de los dos problemas | 20 % |
| Ejecución y explicación del código | 20 % |
| Modificación y análisis de escenarios | 25 % |
| Tablas, ruta, gráficas e interpretación | 20 % |
| Conclusiones y contextualización | 15 % |